In [6]:
import pandas as pd
import numpy as np
import duckdb
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

DATA_DIR = Path("data")
CSV_PATH = DATA_DIR / "/home/gonza/Documentos/Trabajo/Educacion/die/data/raw/estados_descargados/estados_25-9.csv"

In [7]:
df = pd.read_csv(CSV_PATH, dtype=str, keep_default_na=False, na_values=[""])
print("Shape:", df.shape)
print("\nColumnas:")
print(df.dtypes)
df.head(10)
df.columns

Shape: (18130, 12)

Columnas:
codigo                       str
estado                       str
primer_apellido              str
segundo_apellido             str
primer_nombre                str
segundo_nombre               str
dni                          str
doc                          str
obs_reg                      str
obs_10_10                    str
inst_ult_rechazo_marian      str
motivo_ult_rechazo_marian    str
dtype: object


Index(['codigo', 'estado', 'primer_apellido', 'segundo_apellido', 'primer_nombre', 'segundo_nombre', 'dni', 'doc', 'obs_reg', 'obs_10_10', 'inst_ult_rechazo_marian', 'motivo_ult_rechazo_marian'], dtype='str')

In [12]:
# Celda 3: nulos y duplicados
nulos = df.isna().sum().to_frame("nulos")
nulos["%"] = (nulos["nulos"] / len(df) * 100).round(2)
print(nulos.sort_values("%", ascending=False))

print("\nDuplicados por Codigo:", df["codigo"].duplicated().sum())
print("Duplicados por DNI:", df["dni"].duplicated().sum())
print("Filas totalmente duplicadas:", df.duplicated().sum())

                           nulos      %
obs_10_10                  14990  82.68
segundo_apellido            9140  50.41
segundo_nombre              2316  12.77
obs_reg                     2087  11.51
primer_apellido                0   0.00
estado                         0   0.00
codigo                         0   0.00
primer_nombre                  0   0.00
doc                            0   0.00
dni                            0   0.00
inst_ult_rechazo_marian        0   0.00
motivo_ult_rechazo_marian      0   0.00

Duplicados por Codigo: 0
Duplicados por DNI: 18124
Filas totalmente duplicadas: 0


In [ ]:
# Celda 4: categóricas — valores únicos y frecuencias
categoricas = [
    "Estado",
    "Documento",
    "OBSERVACIONES REGULARIDAD",
    "OBSERVACIÓN 01-10 ",
    "Instancia de último Rechazo Marian",
    "Motivo de último Rechazo Marian",
]

for col in categoricas:
    print(f"\n=== {col} ===")
    print(f"Únicos: {df[col].nunique()}")
    print(df[col].value_counts(dropna=False).head(20))

In [10]:
# Celda 5: calidad de DNI y Código
print("DNI — longitud:")
print(df["dni"].dropna().str.len().value_counts().sort_index())

print("\nCódigo — longitud:")
print(df["codigo"].dropna().str.len().value_counts().sort_index())

print("\nCódigo — patrón (primeros 4 chars):")
print(df["codigo"].dropna().str[:4].value_counts().head(10))

DNI — longitud:
dni
2       61
3    18069
Name: count, dtype: int64

Código — longitud:
codigo
15       23
17        2
18        8
19       23
20    17507
21       20
22      547
Name: count, dtype: int64

Código — patrón (primeros 4 chars):
codigo
BM26    17582
BAFT      548
Name: count, dtype: int64


In [13]:
# Celda 6: cruces clave
def crosstab_pct(a, b, top=15):
    ct = pd.crosstab(df[a], df[b], margins=True, margins_name="TOTAL")
    return ct.sort_values("TOTAL", ascending=False).head(top)

print("Estado × Motivo de último Rechazo Marian")
display(crosstab_pct("estado", "motivo_ult_rechazo_marian"))

print("Estado × Instancia de último Rechazo Marian")
display(crosstab_pct("estado", "inst_ult_rechazo_marian"))

print("Estado × OBSERVACIONES REGULARIDAD")
display(crosstab_pct("estado", "obs_reg"))

Estado × Motivo de último Rechazo Marian


motivo_ult_rechazo_marian,#N/D,Baja por otros motivos,NO CORRESPONDE,No Regular,No cumple requisitos de asignación,No matriculado,Otras situaciones,Sin información,TOTAL
estado,,,,,,,,,
TOTAL,420,2,1,12766,1147,1357,34,2403,18130
Baja por desaparición de las causales que justificaron el otorgamiento del beneficio ( ART. 16),0,0,0,6656,0,155,1,1359,8171
Rechazada Art. 5,0,0,0,3570,0,907,18,589,5084
Rechazada Art. 14,0,0,0,2046,493,191,12,398,3140
Rechazada Art. 6,0,0,1,494,654,104,3,57,1313
Baja por percepción de beca,420,0,0,0,0,0,0,0,420
Baja por pase a otro establecimiento (ART. 16),0,1,0,0,0,0,0,0,1
Baja por fallecimiento del beneficiario/a (ART. 16),0,1,0,0,0,0,0,0,1


Estado × Instancia de último Rechazo Marian


inst_ult_rechazo_marian,#N/D,1er Pago,2do Pago,Reco,TOTAL
estado,,,,,
TOTAL,420,6397,8173,3140,18130
Baja por desaparición de las causales que justificaron el otorgamiento del beneficio ( ART. 16),0,0,8171,0,8171
Rechazada Art. 5,0,5084,0,0,5084
Rechazada Art. 14,0,0,0,3140,3140
Rechazada Art. 6,0,1313,0,0,1313
Baja por percepción de beca,420,0,0,0,420
Baja por pase a otro establecimiento (ART. 16),0,0,1,0,1
Baja por fallecimiento del beneficiario/a (ART. 16),0,0,1,0,1


Estado × OBSERVACIONES REGULARIDAD


obs_reg,BECADOS EN OTRO OPERATIVO,CONDICION REGULARIDAD DESCONOCIDA - CRUCE 21- 09,CONDICIÓN NO REGULAR - CRUCE 20 - 08/ REGULAR POR ESCUELA / RECHAZADO DIE,CONDICIÓN REGULARIDAD DESCONOCIDA - CRUCE 21 - 09,Condicion no regular - CRUCE 21-09,DATOS INSUFICIENTES - 22 - 09,DATOS INSUFICIENTES - CRUCE 22 - 09,Datos insuficientes - CRUCE 21 - 09,ESTUDIANTE CON PASE - SIN INFORMACIÓN,LA ESCUELA INFORMA SE DESCONOCE,LA ESCUELA NO ACLARA LA CONDICIÓN,NO COINCIDE CUE - CRUCE 21 - 09,NO ENCONTRADO EN API - CRUCE 21-09,NO ENCONTRADO EN API - CRUCE 22-09 (ESTUDIANTE INFORMADO CON PASE),NO REGULAR,NO REGULAR - CRUCE 13-07,NO REGULAR - CRUCE 23-06,NO REGULAR - INFORMADO POR AGENCIA,NO REGULAR - INFORMADO POR AREA,NO REGULAR - INFORMADO POR AREA/ESCUELA,NO REGULAR SEGÚN INFORMACIÓN ESCUELA,NO-2026-40994583-GCABA-ESC201410.,NO-2026-41717510-GCABA-ESC200808,OTRAS SITUACIONES INFORMADAS POR AREA - ESCUELA,PASE - SIN INFORMACIÓN,RECHAZADA ART. 5 - CRUCE APRENDE BA 3-6,RECHAZADA ART. 6 - CRUCE APRENDE BA 3-6,RECHAZADO DIE,REGULAR - RECHAZADO DIE,SE INFORMA ESTUDIANTE LIBRE POR AREA /ESCUELA,SE INFORMA NO MATRICULADO 1 BIMESTRE,SE INFORMA NO REGULAR POR AREA /ESCUELA,SE INFORMA QUE NO ES ESTUDIANTE DE LA ESCUELA,SE INFORMA SIN DATOS POR PARTE DE LA ESCUELA,SE INFORMÓ PASE,SIN INFORMACION,SIN INFORMACIÓN,SIN INFORMACIÓN - AREA,SIN INFORMACIÓN AREA,SIN MATRICULARSE SEGÚN INFORMACIÓN ESCUELA,VER,VER,VER DECISIÓN DIE,TOTAL
estado,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,,
TOTAL,420,3,8,1,6255,2,385,26,1,16,1,3,35,3,17,117,213,19,57,675,261,1,1,77,13,2867,421,11,2,1,1,100,1,1,2,1931,1713,247,4,117,2,1,11,16043
Baja por desaparición de las causales que justificaron el otorgamiento del beneficio ( ART. 16),0,3,0,1,6255,2,385,26,1,0,0,3,35,3,17,0,0,19,57,0,0,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0,1111,247,4,0,0,0,0,8171
Rechazada Art. 5,0,0,0,0,0,0,0,0,0,16,1,0,0,0,0,117,213,0,0,34,240,0,0,0,10,2867,0,0,0,1,1,92,1,1,1,0,556,0,0,106,0,0,0,4257
Rechazada Art. 14,0,0,8,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,639,0,0,0,77,0,0,0,11,2,0,0,0,0,0,0,1931,0,0,0,0,0,0,0,2668
Rechazada Art. 6,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,2,21,0,0,0,3,0,421,0,0,0,0,8,0,0,1,0,46,0,0,11,2,1,11,527
Baja por percepción de beca,420,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,420


In [16]:
# Celda 7: consistencia
baja_art16 = df["estado"].str.contains("ART. 16", na=False)
sin_motivo = df["motivo_ult_rechazo_marian"].isna() | (df["motivo_ult_rechazo_marian"] == "")
print("Bajas ART.16 sin motivo de rechazo:", (baja_art16 & sin_motivo).sum())
print("Bajas ART.16 con 'Sin información':", (baja_art16 & df["motivo_ult_rechazo_marian"].eq("Sin información")).sum())

Bajas ART.16 sin motivo de rechazo: 0
Bajas ART.16 con 'Sin información': 1359


In [19]:
# Celda 8: agregados para el tablero
agg_estado = df["estado"].value_counts().reset_index()
agg_estado.columns = ["estado", "Cantidad"]

agg_motivo = df["motivo_ult_rechazo_marian"].value_counts().reset_index()
agg_motivo.columns = ["Motivo", "Cantidad"]

cruce_estado_motivo = (
    df.groupby(["estado", "motivo_ult_rechazo_marian"], dropna=False)
      .size().reset_index(name="Cantidad")
)

OUT = Path("outputs"); OUT.mkdir(exist_ok=True)
agg_estado.to_csv(OUT / "agg_estado.csv", index=False)
agg_motivo.to_csv(OUT / "agg_motivo.csv", index=False)
cruce_estado_motivo.to_csv(OUT / "cruce_estado_motivo.csv", index=False)

In [20]:
print(repr(df.columns.tolist()))

['codigo', 'estado', 'primer_apellido', 'segundo_apellido', 'primer_nombre', 'segundo_nombre', 'dni', 'doc', 'obs_reg', 'obs_10_10', 'inst_ult_rechazo_marian', 'motivo_ult_rechazo_marian']


In [21]:
print("COLUMNAS REALES:", [repr(c) for c in df.columns])

COLUMNAS REALES: ["'codigo'", "'estado'", "'primer_apellido'", "'segundo_apellido'", "'primer_nombre'", "'segundo_nombre'", "'dni'", "'doc'", "'obs_reg'", "'obs_10_10'", "'inst_ult_rechazo_marian'", "'motivo_ult_rechazo_marian'"]
